# Lesson 6: Retrieval Critique and Agentic Retry

这一节补上另外两个在 `Agentic RAG` 里非常重要、但前面 notebook 没系统展开的主题：

1. `retrieval critique / self-reflection`
2. `agentic retry policy`

前面的 notebook 已经让 agent 会查资料，但现实里还有一个更难的问题：

- agent 怎么判断“这次查到的证据够不够”？
- 如果不够，下一轮该怎么改写 query，或者换一个工具再查？

这节就专门做一个“会反思、会重试”的检索循环。


## 这一节的技术在做什么

这一节讲的是：

- `retrieval critique / self-reflection`
- `agentic retry policy`

它们解决的是 Agentic RAG 里一个非常现实的问题：

- agent 虽然已经查到了一些内容，但这些内容到底够不够回答问题？

在普通 RAG 里，常见流程是：

1. 检索一次
2. 把结果交给模型
3. 直接回答

这种方式简单，但也很脆弱。因为现实里经常出现：

- 检索结果只覆盖了问题的一半
- 文档虽然相关，但不够具体
- 工具选对了，但 query 问得不够好
- 第一次证据不足，却没有触发第二轮补查

所以更成熟的 Agentic RAG 系统通常不会把“第一次检索结果”直接当成最终依据，而是会加一层 `critique`：

- 当前证据支持了什么？
- 还缺什么？
- 置信度高不高？
- 要不要再查一次？

这就是 `retrieval critique` 的作用。

`self-reflection` 则更强调：

- agent 不只是执行检索
- 还会回头审视自己的检索是否足够好

`agentic retry policy` 再往前走一步，它回答的是：

- 如果证据不够，下一轮该怎么查？
- 是继续用同一个工具？
- 还是换一个工具？
- 是保留原问题？
- 还是要改写 query？

所以这一节其实是在补上 Agentic RAG 的“纠错层”。

你可以把它理解成：

- `Lesson 5` 解决“怎么规划”
- `Lesson 6` 解决“查得不够好时，怎么反思并重试”

这也是很多真实 agent 系统和简单 demo 的区别：

- demo：查一次就答
- 系统：会判断查得够不够，不够就继续补查

这一节最重要的收获是：

- `检索` 不应该只是一次动作
- 它应该是一个“可评估、可反思、可重试”的循环


## Setup


In [ ]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Any, Optional

import nest_asyncio
from pydantic import BaseModel, Field

nest_asyncio.apply()


In [ ]:
# 这一组 notebook 不是重新造一套数据，而是直接复用 Lesson 4 里的论文和工具工厂。
# 这样你学到的是“在原有 Agentic RAG 系统上继续加高级模式”。

lesson4_dir = (Path.cwd() / "../Lesson_4").resolve()
if str(lesson4_dir) not in sys.path:
    sys.path.append(str(lesson4_dir))

from helper import get_dashscope_api_key
from utils import get_doc_tools

from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai_like import OpenAILike


def find_workspace_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if candidate.name == "AIAgent":
            return candidate
    raise FileNotFoundError("Could not find the AIAgent workspace root.")


def find_local_bge_snapshot() -> Path:
    workspace_root = find_workspace_root()
    snapshot_root = workspace_root / "models" / "models--BAAI--bge-small-en-v1.5" / "snapshots"
    snapshots = sorted(
        path for path in snapshot_root.iterdir()
        if path.is_dir() and (path / "config.json").exists()
    )
    if not snapshots:
        raise FileNotFoundError(f"No valid local BGE snapshot found under {snapshot_root}")
    return snapshots[0]


# 继续沿用课程中的 qwen-max + 本地 BGE embedding 配置。
llm = OpenAILike(
    api_key=get_dashscope_api_key(),
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-max",
    temperature=0.1,
    context_window=128000,
    is_chat_model=True,
    is_function_calling_model=True,
)
Settings.llm = llm
Settings.embed_model = HuggingFaceEmbedding(
    model_name=str(find_local_bge_snapshot()),
    device="cpu",
)


papers = [
    "metagpt.pdf",
    "longlora.pdf",
    "selfrag.pdf",
]


def build_tool_registry() -> dict[str, Any]:
    registry = {}
    for paper in papers:
        paper_name = Path(paper).stem
        paper_path = lesson4_dir / paper
        vector_tool, summary_tool = get_doc_tools(str(paper_path), paper_name)
        registry[vector_tool.metadata.name] = vector_tool
        registry[summary_tool.metadata.name] = summary_tool
    return registry


tool_registry = build_tool_registry()
tool_names = list(tool_registry.keys())
print("Loaded tools:", tool_names)


def extract_json_object(text: str) -> dict[str, Any]:
    # qwen-max 有时会返回 ```json 代码块，有时直接返回 JSON。
    # 这里统一把最外层 JSON 对象提出来，再交给 Pydantic 做结构校验。
    cleaned = text.strip()
    cleaned = cleaned.replace("```json", "```").replace("```JSON", "```")
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`").strip()
    match = re.search(r"\{.*\}", cleaned, re.S)
    if not match:
        raise ValueError(f"Could not find JSON object in model output:\n{cleaned}")
    return json.loads(match.group(0))


def complete_json(prompt: str) -> dict[str, Any]:
    response = llm.complete(prompt)
    return extract_json_object(response.text)


def call_tool(tool_name: str, query: str, page_numbers: Optional[list[str]] = None) -> dict[str, Any]:
    # 课程里的工具分成两类：
    # 1. vector_tool_* 需要 query，可选 page_numbers
    # 2. summary_tool_* 通常只需要一个输入问题
    tool = tool_registry[tool_name]
    page_numbers = page_numbers or []

    attempts: list[dict[str, Any]]
    if tool_name.startswith("vector_tool_"):
        attempts = [
            {"query": query, "page_numbers": page_numbers},
            {"query": query},
            {"input": query},
        ]
    else:
        attempts = [
            {"input": query},
            {"query": query},
        ]

    last_error = None
    for kwargs in attempts:
        try:
            tool_output = tool.call(**kwargs)
            raw_output = getattr(tool_output, "raw_output", None)
            source_nodes = getattr(raw_output, "source_nodes", []) if raw_output is not None else []
            source_pages = [
                node.metadata.get("page_label")
                for node in source_nodes
                if hasattr(node, "metadata")
            ]
            return {
                "tool_name": tool_name,
                "query": query,
                "content": str(getattr(tool_output, "content", tool_output)),
                "source_pages": [p for p in source_pages if p],
            }
        except Exception as exc:  # noqa: BLE001
            last_error = exc

    raise RuntimeError(f"Tool call failed for {tool_name}: {last_error}")


def build_tool_catalog() -> str:
    # 这份目录会喂给 planner / critic / evaluator。
    # 它相当于让模型先知道“我手上有哪些工具，每个工具大概适合干什么”。
    lines = []
    for tool_name, tool in tool_registry.items():
        lines.append(f"- {tool_name}: {tool.metadata.description}")
    return "\n".join(lines)


tool_catalog = build_tool_catalog()


## 1. Define critique and retry schemas


In [ ]:
class RetrievalCritique(BaseModel):
    is_sufficient: bool = Field(description="Whether the current evidence is enough to answer the question.")
    confidence: float = Field(description="A confidence score from 0 to 1.")
    missing_information: list[str] = Field(description="What is still missing from the evidence.")
    strengths: list[str] = Field(description="What the current evidence already supports well.")


class RetryDecision(BaseModel):
    should_retry: bool = Field(description="Whether another retrieval round is needed.")
    suggested_tool_name: str = Field(description="Which tool should be used next.")
    rewritten_query: str = Field(description="A better follow-up query.")
    reason: str = Field(description="Why this retry strategy is appropriate.")


In [ ]:
def initial_retrieval(question: str) -> dict[str, Any]:
    # 为了突出“critique + retry”，这里故意先做一个比较朴素的初始检索：
    # 让模型先选一个最可能的工具，而不是一上来就做完整多步规划。
    prompt = f'''
    You are choosing the best first retrieval tool.

    Available tools:
    {tool_catalog}

    User question:
    {question}

    Return JSON only:
    {{
      "tool_name": "...",
      "query": "...",
      "reason": "..."
    }}
    '''
    decision = complete_json(prompt)
    result = call_tool(decision["tool_name"], decision["query"])
    result["reason"] = decision["reason"]
    return result


def critique_retrieval(question: str, current_evidence: str) -> RetrievalCritique:
    # critique 阶段不直接继续检索，而是先审视“当前证据到底能不能回答问题”。
    prompt = f'''
    You are critiquing retrieval evidence for an agentic RAG system.

    User question:
    {question}

    Current evidence:
    {current_evidence}

    Return JSON only:
    {{
      "is_sufficient": true,
      "confidence": 0.0,
      "missing_information": ["..."],
      "strengths": ["..."]
    }}
    '''
    critique_dict = complete_json(prompt)
    return RetrievalCritique.model_validate(critique_dict)


def create_retry_decision(question: str, critique: RetrievalCritique, previous_trace: list[dict[str, Any]]) -> RetryDecision:
    # retry policy 的重点不是“盲目再查一次”，而是根据 critique 结果有方向地调整。
    previous_summary = "\n".join(
        f"- Tool={item['tool_name']} | Query={item['query']}"
        for item in previous_trace
    )
    prompt = f'''
    You are deciding how an agentic RAG system should retry retrieval.

    Available tools:
    {tool_catalog}

    User question:
    {question}

    Critique:
    {critique.model_dump_json(indent=2)}

    Previous retrieval attempts:
    {previous_summary}

    Return JSON only:
    {{
      "should_retry": true,
      "suggested_tool_name": "...",
      "rewritten_query": "...",
      "reason": "..."
    }}
    '''
    retry_dict = complete_json(prompt)
    return RetryDecision.model_validate(retry_dict)


## 2. Build a reflective retrieval loop


In [ ]:
def synthesize_answer_from_trace(question: str, trace: list[dict[str, Any]]) -> str:
    evidence_text = "\n\n".join(
        f"Round {item['round']} | Tool={item['tool_name']} | Query={item['query']}\n{item['content']}"
        for item in trace
    )
    prompt = f'''
    You are answering from a reflective retrieval trace.

    User question:
    {question}

    Retrieval trace:
    {evidence_text}

    Write a grounded answer using only the retrieved evidence.
    Mention any uncertainty that still remains.
    '''
    return llm.complete(prompt).text


def run_reflective_agentic_rag(question: str, max_rounds: int = 3) -> dict[str, Any]:
    trace: list[dict[str, Any]] = []

    # Round 1：先做一次初始检索。
    first_result = initial_retrieval(question)
    trace.append(
        {
            "round": 1,
            "tool_name": first_result["tool_name"],
            "query": first_result["query"],
            "content": first_result["content"],
            "reason": first_result["reason"],
            "source_pages": first_result["source_pages"],
        }
    )

    for round_id in range(1, max_rounds + 1):
        current_evidence = "\n\n".join(item["content"] for item in trace)
        critique = critique_retrieval(question, current_evidence)

        # 一旦 critic 认为证据已经够了，就停止重试。
        if critique.is_sufficient:
            answer = synthesize_answer_from_trace(question, trace)
            return {
                "trace": trace,
                "critique": critique,
                "final_answer": answer,
            }

        retry = create_retry_decision(question, critique, trace)
        if not retry.should_retry:
            answer = synthesize_answer_from_trace(question, trace)
            return {
                "trace": trace,
                "critique": critique,
                "final_answer": answer,
            }

        retry_result = call_tool(retry.suggested_tool_name, retry.rewritten_query)
        trace.append(
            {
                "round": len(trace) + 1,
                "tool_name": retry.suggested_tool_name,
                "query": retry.rewritten_query,
                "content": retry_result["content"],
                "reason": retry.reason,
                "source_pages": retry_result["source_pages"],
            }
        )

    answer = synthesize_answer_from_trace(question, trace)
    return {
        "trace": trace,
        "critique": critique_retrieval(question, "\n\n".join(item["content"] for item in trace)),
        "final_answer": answer,
    }


In [ ]:
# 这个问题故意设计成需要“先找到论文中的评估数据集，再补充结果和差异”。
reflective_question = (
    "Tell me about the evaluation datasets used in Self-RAG and LongLoRA, "
    "and explain whether they are being used for the same kind of claim."
)

reflective_run = run_reflective_agentic_rag(reflective_question, max_rounds=3)
reflective_run["critique"]


In [ ]:
for item in reflective_run["trace"]:
    print(f"Round {item['round']}")
    print(f"Tool: {item['tool_name']}")
    print(f"Query: {item['query']}")
    print(f"Reason: {item['reason']}")
    print(f"Source pages: {item['source_pages']}")
    print(item["content"][:1000])
    print("-" * 100)


In [ ]:
print(reflective_run["final_answer"])
